# Testing Methods in VS Code Notebook

This notebook shows alternative ways to test code quickly and reliably.
It is written so you can adapt the same pattern to `llava_descriptor.py` and other modules.

## 1) Set Up a Small Function to Test

Define a simple function in a notebook cell (or import from a module) to use as the test target.

In [1]:
def classify_trip_duration(seconds: int) -> str:
    if seconds < 0:
        return "invalid"
    if seconds < 300:
        return "short"
    if seconds <= 1800:
        return "medium"
    return "long"

classify_trip_duration(120)

'short'

## 2) Run Quick Checks in Notebook Cells

Execute sample inputs and inspect outputs interactively.

In [2]:
samples = [-5, 0, 120, 500, 2000]
for s in samples:
    print(f"{s:>4} -> {classify_trip_duration(s)}")

  -5 -> invalid
   0 -> short
 120 -> short
 500 -> medium
2000 -> long


## 3) Add Assertions for Repeatable Checks

Use `assert` to quickly catch regressions.

In [3]:
assert classify_trip_duration(-1) == "invalid"
assert classify_trip_duration(10) == "short"
assert classify_trip_duration(900) == "medium"
assert classify_trip_duration(4000) == "long"
print("Assertions passed ✅")

Assertions passed ✅


## 4) Create `pytest` Unit Tests in a Test File

Generate a test file automatically from notebook to keep checks reusable in CI.

In [4]:
from pathlib import Path

test_file = Path("tests/test_duration_notebook_generated.py")
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text(
    """
from src.scoring.leader_scorer import clamp01


def test_clamp01_bounds():
    assert clamp01(-1.5) == 0.0
    assert clamp01(0.5) == 0.5
    assert clamp01(3.0) == 1.0
""".strip() + "\n",
    encoding="utf-8",
)
print(f"Created: {test_file}")

Created: tests\test_duration_notebook_generated.py


## 5) Run Tests in VS Code Test Explorer

In VS Code:
1. Open Testing panel.
2. Discover tests in `tests/`.
3. Run or debug tests individually.

You can also run quickly in a notebook cell with `pytest`.

In [5]:
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


=================================== ERRORS ====================================
_________ ERROR collecting tests/test_duration_notebook_generated.py __________
ImportError while importing test module 'c:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour\docs\tests\test_duration_notebook_generated.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
C:\Users\redaj\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests\test_duration_notebook_generated.py:1: in <module>
    from src.scoring.leader_scorer import clamp01
E   ModuleNotFoundError: No module named 'src'
=========================== short test summary info ===========================
ERROR tests/test_duration_notebook_generated.py
!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.49s




## 6) Use `doctest` for Inline Examples

Doctest keeps doc examples executable and validated.

In [6]:
def add(a: int, b: int) -> int:
    """
    Add two integers.

    >>> add(2, 3)
    5
    >>> add(-1, 1)
    0
    """
    return a + b

import doctest
_ = doctest.testmod(verbose=True)

Trying:
    add(2, 3)
Expecting:
    5
ok
Trying:
    add(-1, 1)
Expecting:
    0
ok
2 items had no tests:
    __main__
    __main__.classify_trip_duration
1 items passed all tests:
   2 tests in __main__.add
2 tests in 3 items.
2 passed and 0 failed.
Test passed.


## 7) Optional: Add Property-Based Tests with `hypothesis`

Use generated random inputs to test invariants.

> Install first if needed: `pip install hypothesis`

In [7]:
try:
    from hypothesis import given, strategies as st

    @given(st.integers(), st.integers())
    def test_add_commutative(a, b):
        assert add(a, b) == add(b, a)

    test_add_commutative()
    print("Hypothesis sample test passed ✅")
except Exception as exc:
    print("Hypothesis not available or test skipped:", exc)

Hypothesis not available or test skipped: No module named 'hypothesis'


## LLaVA Direct Test (Practical)

Use this cell to run a strict LLaVA test directly from notebook.
Pass criteria:
- command exit code is `0`
- summary JSON exists with `descriptions_written > 0`
- generated text is not the mock template

In [8]:
import json
import subprocess
import sys
from pathlib import Path

summary_path = Path("outputs/descriptions/llava_direct_summary.json")
jsonl_path = Path("outputs/descriptions/llava_direct_descriptions.jsonl")

cmd = [
    sys.executable,
    "src/description/llava_descriptor.py",
    "--source", "outputs/reid_tracking_smoke/synthetic_input.mp4",
    "--tracks-jsonl", "outputs/reid_tracking_smoke/test_run_auto/tracks.jsonl",
    "--output-jsonl", str(jsonl_path),
    "--output-summary", str(summary_path),
    "--camera-id", "cam_llava_nb",
    "--backend", "llava",
    "--strict-llava",
    "--device", "cuda",
    "--sample-every", "10",
    "--max-events", "5",
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print("exit code:", proc.returncode)
print("STDOUT:\n", proc.stdout[:4000])
print("STDERR:\n", proc.stderr[:4000])

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("summary:", summary)

if jsonl_path.exists():
    lines = jsonl_path.read_text(encoding="utf-8").splitlines()[:3]
    for i, line in enumerate(lines, 1):
        print(f"line {i}: {line[:400]}")

exit code: 2
STDOUT:
 
STDERR:
 C:\Users\redaj\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\redaj\\Desktop\\ME\\THE-ONE\\human_behaviour\\docs\\src\\description\\llava_descriptor.py': [Errno 2] No such file or directory

